#### Vector Store DataBases

1. FAISS
2. Chroma DB
3. Astro DB ( This we will see later, when we use cloud and all )

Here we can see all the vector databases we can use with langchain, how many they have integrated and all
https://python.langchain.com/v0.2/docs/integrations/vectorstores/

#### 1. FAISS

Facebook AI Similarity Search (Faiss) is a library for efficient similarity search and clustering of dense vectors. It contains algorithms that search in sets of vectors of any size, up to ones that possibly do not fit in RAM. It also contains supporting code for evaluation and parameter tuning.

The steps are same:
1. Load the documents 
2. Split them 
3. Convert to embeddings
4. Store them in DB

In [1]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import CharacterTextSplitter

loader = TextLoader("Data/speech.txt")
documents = loader.load()
text_splitter=CharacterTextSplitter(chunk_size=1000, chunk_overlap =30)
docs = text_splitter.split_documents(documents)


/home/dragon/.virtualenvs/langchain/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
docs

[Document(metadata={'source': 'Data/speech.txt'}, page_content='The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no indemnities for ourselves, no material compensation for the sacrifices we shall freely make. We are but one of the champions of the rights of mankind. We shall be satisfied when those rights have been made as secure as the faith and the freedom of nations can make them.\n\nJust because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.\n\n…'),
 Document(metadata={'source': 'Data/speech.txt'}, page_content='…\n\nIt will be all the easier for us to c

In [3]:
embeddings = OllamaEmbeddings(model ="mxbai-embed-large")

db = FAISS.from_documents(docs, embeddings)
db


/tmp/ipykernel_144591/4247703218.py:1: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(model ="mxbai-embed-large")


Now, that we have stored all the documents in the db , we can query anything ( basically read the speech.txt and form some relevant questions about it)

In [10]:
query = "how does the speaker think about war "
docs = db.similarity_search(query)
docs  #This will show all the matching documents with the highest matching one at the top 

[Document(id='6c71418d-02be-4dc8-b0d0-99482871cb69', metadata={'source': 'Data/speech.txt'}, page_content='…\n\nIt will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them, but only in armed opposition to an irresponsible government which has thrown aside all considerations of humanity and of right and is running amuck. We are, let me say again, the sincere friends of the German people, and shall desire nothing so much as the early reestablishment of intimate relations of mutual advantage between us—however hard it may be for them, for the time being, to believe that this is spoken from our hearts.'),
 Document(id='acf070b1-ad9e-4462-b42e-a6cdaa0739ba', metadata={'source': 'Data/speech.txt'}, page_content='The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political

In [11]:
# We will display the top one
docs[0]

Document(id='6c71418d-02be-4dc8-b0d0-99482871cb69', metadata={'source': 'Data/speech.txt'}, page_content='…\n\nIt will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them, but only in armed opposition to an irresponsible government which has thrown aside all considerations of humanity and of right and is running amuck. We are, let me say again, the sincere friends of the German people, and shall desire nothing so much as the early reestablishment of intimate relations of mutual advantage between us—however hard it may be for them, for the time being, to believe that this is spoken from our hearts.')

##### As a Retriever
We can also convert the vectorstore into a Retriever class. This allows us to easily use it in other LangChain methods, which largely work with retrievers

In [13]:
retriever = db.as_retriever()

docs = retriever.invoke(query) #Here also we can achive the same result, but this way we will use in future for diff types of models and all
docs[0]

Document(id='6c71418d-02be-4dc8-b0d0-99482871cb69', metadata={'source': 'Data/speech.txt'}, page_content='…\n\nIt will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them, but only in armed opposition to an irresponsible government which has thrown aside all considerations of humanity and of right and is running amuck. We are, let me say again, the sincere friends of the German people, and shall desire nothing so much as the early reestablishment of intimate relations of mutual advantage between us—however hard it may be for them, for the time being, to believe that this is spoken from our hearts.')

In [ ]:
docs[0].page_content #Here we can query the page_content attribute of the document object

'…\n\nIt will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them, but only in armed opposition to an irresponsible government which has thrown aside all considerations of humanity and of right and is running amuck. We are, let me say again, the sincere friends of the German people, and shall desire nothing so much as the early reestablishment of intimate relations of mutual advantage between us—however hard it may be for them, for the time being, to believe that this is spoken from our hearts.'

##### Similarity Search with score
There are some FAISS specific methods. One of them is similarity_search_with_score, which allows you to return not only the documents but also the distance score of the query to them. The returned distance score is L2 distance. Therefore, a lower score is better.

In [16]:
docs_and_score = db.similarity_search_with_score(query)
docs_and_score

[(Document(id='6c71418d-02be-4dc8-b0d0-99482871cb69', metadata={'source': 'Data/speech.txt'}, page_content='…\n\nIt will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them, but only in armed opposition to an irresponsible government which has thrown aside all considerations of humanity and of right and is running amuck. We are, let me say again, the sincere friends of the German people, and shall desire nothing so much as the early reestablishment of intimate relations of mutual advantage between us—however hard it may be for them, for the time being, to believe that this is spoken from our hearts.'),
  np.float32(0.67197657)),
 (Document(id='acf070b1-ad9e-4462-b42e-a6cdaa0739ba', metadata={'source': 'Data/speech.txt'}, page_content='The world must be made safe for democracy. Its peace must be planted upon the te

We can also pass vectors directly instead of sentences

--> For that first we need to create a vector and then give it as query


In [17]:
embedding_vector = embeddings.embed_query(query)
embedding_vector

[0.029234865680336952,
 -0.01022513397037983,
 0.016739828512072563,
 0.00976596586406231,
 -0.008866746909916401,
 -0.08743452280759811,
 -0.026683827862143517,
 0.015439952723681927,
 -0.020412879064679146,
 -0.0019503559451550245,
 0.0031821129377931356,
 -0.008127882145345211,
 0.001394862774759531,
 0.005212115123867989,
 -0.008283701725304127,
 -0.018249843269586563,
 -0.045824337750673294,
 -0.006648666691035032,
 0.003993427846580744,
 -0.00904315896332264,
 -0.008178061805665493,
 0.0003447740455158055,
 -0.010205790400505066,
 0.03075113147497177,
 -0.025455771014094353,
 0.019010964781045914,
 -0.0479581318795681,
 0.036510150879621506,
 0.06599272042512894,
 0.02311522886157036,
 0.011022435501217842,
 0.0009568401728756726,
 -0.023612819612026215,
 -0.06328088790178299,
 0.005045394413173199,
 -0.047985631972551346,
 0.03210834413766861,
 -0.09397418051958084,
 -0.022145120427012444,
 0.004371797665953636,
 0.0088141318410635,
 0.017191119492053986,
 0.03855004161596298,
 

In [18]:
docs_with_score = db.similarity_search_by_vector(embedding_vector)
docs_with_score

[Document(id='6c71418d-02be-4dc8-b0d0-99482871cb69', metadata={'source': 'Data/speech.txt'}, page_content='…\n\nIt will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them, but only in armed opposition to an irresponsible government which has thrown aside all considerations of humanity and of right and is running amuck. We are, let me say again, the sincere friends of the German people, and shall desire nothing so much as the early reestablishment of intimate relations of mutual advantage between us—however hard it may be for them, for the time being, to believe that this is spoken from our hearts.'),
 Document(id='acf070b1-ad9e-4462-b42e-a6cdaa0739ba', metadata={'source': 'Data/speech.txt'}, page_content='The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political

##### Saving and Loading the DB

In [ ]:
db.save_local("Faiss_Index_DB") #This will create a folder with the given name and save the database in that

In [20]:
#Now after saving, we can also load the local db
new_db = FAISS.load_local("Faiss_Index_DB", embeddings, allow_dangerous_deserialization=True) 
# This additional dangerous parameter we will use when we get some pickle serialization errors

In [21]:
docs_new = new_db.similarity_search(query)
docs_new

[Document(id='6c71418d-02be-4dc8-b0d0-99482871cb69', metadata={'source': 'Data/speech.txt'}, page_content='…\n\nIt will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them, but only in armed opposition to an irresponsible government which has thrown aside all considerations of humanity and of right and is running amuck. We are, let me say again, the sincere friends of the German people, and shall desire nothing so much as the early reestablishment of intimate relations of mutual advantage between us—however hard it may be for them, for the time being, to believe that this is spoken from our hearts.'),
 Document(id='acf070b1-ad9e-4462-b42e-a6cdaa0739ba', metadata={'source': 'Data/speech.txt'}, page_content='The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political

#### 2. Chroma
Chroma is a AI-native open-source vector database focused on developer productivity and happiness. Chroma is licensed under Apache 2.0.

In [ ]:
#Building a sample vector DB
#Here we will directly use the new library called langchain_chroma directly instead of getting from chromadb separately which we did earlier
from langchain_chroma import Chroma
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import OllamaEmbeddings


#Step 1: load
loader = TextLoader("Data/speech.txt")
data = loader.load()
data

[Document(metadata={'source': 'Data/speech.txt'}, page_content='The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no indemnities for ourselves, no material compensation for the sacrifices we shall freely make. We are but one of the champions of the rights of mankind. We shall be satisfied when those rights have been made as secure as the faith and the freedom of nations can make them.\n\nJust because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.\n\n…\n\nIt will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairne

In [ ]:
#Step2: Split
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap = 30)
splits = text_splitter.split_documents(data)
splits

[Document(metadata={'source': 'Data/speech.txt'}, page_content='The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no indemnities for ourselves, no material compensation for the sacrifices we shall freely make. We are but one of the champions of the rights of mankind. We shall be satisfied when those rights have been made as secure as the faith and the freedom of nations can make them.'),
 Document(metadata={'source': 'Data/speech.txt'}, page_content='Just because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.\n\n…'),
 Document(metadata={'source': 'Data/speech

In [26]:
#Step3: Convert to embeddings and store in the database ( here both can be done in single step itself)
embeddings= OllamaEmbeddings(model = "mxbai-embed-large")
vector_db = Chroma.from_documents(documents=splits, embedding=embeddings)
vector_db


In [30]:
#Step 4: Query any text to the DB to get the relevant results by vector similarity search
query = "what does the speaker believe is the main reason united states should enter the war?"
docs = vector_db.similarity_search(query)
docs[0].page_content

'It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our hearts—for democracy,'

We can also save this chroma DB to the local.

In [39]:
##Saving to disk, to do this we just give additonal parameter called persist_directory
vector_db =Chroma.from_documents(documents=splits, embedding=embeddings, persist_directory="Chroma_local_DB")

We can load it again by giving the directory name

In [42]:
db_2 = Chroma(persist_directory="Chroma_local_DB", embedding_function=embeddings)
docs_new_2 = db_2.similarity_search(query)
docs_new_2[0].page_content

'It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our hearts—for democracy,'

Here also we can perform similarity search with score

In [43]:
docs_with_score = db_2.similarity_search_with_score(query)
docs_with_score[0]

(Document(id='df273744-04e9-42bd-a8f3-fb3e789092e8', metadata={'source': 'Data/speech.txt'}, page_content='It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our hearts—for democracy,'),
 0.6951611042022705)

Here also we can use the db as retriever

In [45]:
retriever = db_2.as_retriever()
retriever.invoke(query)[0].page_content

'It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our hearts—for democracy,'